In [25]:
import numpy as np
import proplot as pplt
from pathlib import Path
from utils.data_processing import remove_padding
import cartopy.feature as cfeature

In [26]:
real_path = your_file_path
pred_path = your_file_path
real = np.load(real_path)['input']
pred = np.load(pred_path)['input']
mask_file = f'your_directory/bucket/dnstream/stats/pmask.npz'
pmask = np.load(mask_file, allow_pickle=True)['pmask']
real = remove_padding(real, pmask)
pred = remove_padding(pred, pmask)

coordinate_path = "your_directory/bucket/dnstream/stats/coordinate.npz"
coor_data = np.load(coordinate_path)
lats, lons = coor_data['lat'], coor_data['lon']
lon2d, lat2d = np.meshgrid(lons, lats)

map_mask_file = f'your_directory/bucket/dnstream/stats/map_mask.npz'
output_mask = np.load(map_mask_file, allow_pickle=True)['input_mask']
output_mask = remove_padding(output_mask, pmask)
data1 = output_mask[3]

In [ ]:
diff = pred - real
n_channels = real.shape[0]
fig, axs = pplt.subplots(nrows=n_channels, ncols=3, figsize=(9, 2 * n_channels), share=False, proj='cyl',wspace=0.2,
                         lonlim=(lons[0], lons[-1]), latlim=(lats[0], lats[-1]), dpi=300)
fig.patch.set_facecolor('white')

channel_names = [
    'u-component of wind @80m (m/s)',           # 'ugrd'
    'v-component of wind @80m (m/s)',           # 'vgrd'
    'Relative Humidity × 100 (%)',              # relative_humidity
    'Surface Pressure (hPa)',              # surface_pressure
    'Total Precipitable Water (kg/m²)',   # total_precipitable_water
    'Air Temperature × 10 (°C)',               # air_temperature
    'Downward Longwave Radiation (W/m²)', # GLW
    'Downward Shortwave Radiation (W/m²)']


for i in range(n_channels):
    datacache1 = real[i].copy()
    datacache2 = real[i].copy()
    datacache1[data1 == 0] = np.nan
    datacache2[data1 == 0] = np.nan
    vmin = np.nanpercentile([datacache1, datacache2], 0.1)
    if vmin==-1:
        vmin=0
    vmax = np.nanpercentile([datacache1, datacache2], 99.9)
    absmax = np.nanmax(np.abs(diff[i]))
    absmin = np.nanmin(diff[i])
    for j in range(3):
        ax = axs[i, j]
        ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.5)
        ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.5)
        ax.add_feature(cfeature.STATES.with_scale('50m'), linewidth=0.5)
        if j == 0:
            data = real[i].copy()
            cmap = 'viridis'
            levels = np.linspace(vmin, vmax, 21)
        elif j == 1:
            data = pred[i].copy()
            cmap = 'viridis'
            levels = np.linspace(vmin, vmax, 21)
        else:
            data = diff[i].copy()
            cmap = 'bwr'
            levels = np.linspace(absmin, absmax, 21)
        data[data1 == 0] = np.nan
        m = ax.pcolormesh(lon2d, lat2d, data, cmap=cmap, levels=levels, extend='both')
        ax.format(lonlines=10, latlines=10)
        if j == 0:
            ax.colorbar(m, loc='b',label='Real', length=0.8, width=0.1)
            ax.format(title=channel_names[i], titleloc='l',titlesize=8)
        if j == 1:
            ax.colorbar(m, loc='b',label='Predicted', length=0.8, width=0.1)
        if j == 2:
            ax.colorbar(m, loc='b',label='Difference', length=0.8, width=0.1)
fig.show()
fig.savefig("./sifigs/downscale_example.png", bbox_inches='tight', pad_inches=0)

/home/cfeng/.conda/envs/pybkb/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:403: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "
/home/cfeng/.conda/envs/pybkb/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:403: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "
/home/cfeng/.conda/envs/pybkb/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:403: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  warnings.warn("The `map_projection` keyword argument is "
/home/cfeng/.conda/envs/pybkb/lib/python3.11/site-packages/cartopy/mpl/geoaxes.py:403: UserWarning: The `map_projection` keyword argument is deprecated, use `projection` to instantiate a GeoAxes instead.
  wa